#Imports

In [ ]:
!pip install woodelf_explainer==0.2.12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.5 MB/s eta 0:00:00


In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
from typing import Union, Dict, Optional, Tuple, Set, List
from math import factorial
import time
from copy import copy
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score
import scipy
import shap

import lightgbm as lgb

# For GPU execution
# import cupy as cp

In [ ]:
from woodelf.parse_models import load_decision_tree_ensemble_model
from woodelf.cube_metric import CubeMetric, ShapleyInteractionValues, ShapleyValues

import woodelf


In [ ]:
# Useful if you run this on google colab and downloaded the data into your drive.
# If you run the notebook in other environment remove these lines and change the 'pd.read_csv()' function in this notebook to read from
# where you saved you data
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Environment Note
All the code below, that works with the EEEI-CIS fraud data will run perfectly in the free version of Google Colab.
The free version machine have 12GB RAM and this is enough for this part of the notebook.

Next, we will load the KDD-Cup dataset. As it includes millions of rows we need more than 12GB to run the algorithm.
This part of the notebook needs the 50GB CPU machine. It is available for subscribed members for Google Colab (toggle the High-RAM option in Runtime->Change runtime type button).

If you only bought computation units, but not a subscription, you can still run the full notebook using the v2-8-TPU machine. This machine have more then enough RAM and it is pretty cheap. Its CPU is a bit slower though, so you will get a slower runtimes.

Final note: preformance in Colab depends on the machine allocated (the CPU option can allocate several types of machines) and on load balancing inside Colab (when more users use the system running times might be slower). So the running times of our algorithms and the shap python package can very. The difference between our approach and the current state-of-the-art is big enough to be noticed, but the excat running times can be slightly different from the ones stated our paper.

# Fraud Data Preprocessing and Model training

In [ ]:
transactions_train = pd.read_parquet('drive/MyDrive/ShapResearch/HighDepth/IJCAI/data/ieee_cis_fraud_train.parquet') # columns are train_features + ['isFraud']
transactions_test = pd.read_parquet('drive/MyDrive/ShapResearch/HighDepth/IJCAI/data/ieee_cis_fraud_test.parquet') # columns are train_features + ['isFraud']

train_features = [f for f in transactions_train.columns if f != 'isFraud']
fraud_train = transactions_train[train_features]
fraud_test = transactions_test[train_features]

In [ ]:
LIGHTGBM_PARAMS = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.1,

    # Allow high depth and enough leaves to reach this possible depth
    "num_leaves": 2024,
    "max_depth": 10,
    "min_data_in_leaf": 500,        # Does provide some regulation

    # Sampling (stability + reduces overfit)
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,

    # Practical
    "verbosity": -1,
    "seed": 42,
    "force_col_wise": True,            # often faster/safer for wide data
}


def print_model_stat(model, features):
    mobj = load_decision_tree_ensemble_model(model, features)
    depths = {}
    for tree in mobj.trees:
        for leaf, path in tree.get_all_leaves_with_paths():
            depth = len(path)
            if depth not in depths:
                depths[depth] = 0
            depths[depth] += 1

    for depth in sorted(list(depths.keys())):
        print(f"\tPaths of depth {depth}: {depths.get(depth, 0)} paths")

def lightgbm_model(X_train, y_train, params, num_rounds=100):
    train_set = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
    return lgb.train(
        params=params,
        train_set=train_set,
        num_boost_round=num_rounds
    )


def different_depth_lightgbm_models(trainset, y, params, num_rounds, depths):
    models = {}
    for depth in depths:
        new_params = params.copy()
        new_params['max_depth'] = depth
        models[depth] = lightgbm_model(trainset, y, new_params, num_rounds=num_rounds)
        print("\n\n")
        print(f"Trained on depth {depth}")
        print_model_stat(models[depth], list(trainset.columns))
    return models

In [ ]:
gbm_100trees = different_depth_lightgbm_models(transactions_train[train_features], transactions_train['isFraud'], LIGHTGBM_PARAMS, num_rounds=100, depths=[6,9,12,15,18])




Trained on depth 6
	Paths of depth 1: 6 paths
	Paths of depth 2: 60 paths
	Paths of depth 3: 135 paths
	Paths of depth 4: 284 paths
	Paths of depth 5: 473 paths
	Paths of depth 6: 2086 paths



Trained on depth 9
	Paths of depth 1: 16 paths
	Paths of depth 2: 55 paths
	Paths of depth 3: 138 paths
	Paths of depth 4: 258 paths
	Paths of depth 5: 454 paths
	Paths of depth 6: 710 paths
	Paths of depth 7: 959 paths
	Paths of depth 8: 1222 paths
	Paths of depth 9: 3752 paths



Trained on depth 12
	Paths of depth 1: 15 paths
	Paths of depth 2: 55 paths
	Paths of depth 3: 143 paths
	Paths of depth 4: 290 paths
	Paths of depth 5: 472 paths
	Paths of depth 6: 636 paths
	Paths of depth 7: 884 paths
	Paths of depth 8: 1140 paths
	Paths of depth 9: 1496 paths
	Paths of depth 10: 1681 paths
	Paths of depth 11: 1991 paths
	Paths of depth 12: 4782 paths



Trained on depth 15
	Paths of depth 1: 16 paths
	Paths of depth 2: 72 paths
	Paths of depth 3: 121 paths
	Paths of depth 4: 278 paths
	Paths of

In [ ]:
gbm_10trees = different_depth_lightgbm_models(transactions_train[train_features], transactions_train['isFraud'], LIGHTGBM_PARAMS, num_rounds=10, depths=[12,15,18, 21])




Trained on depth 12
	Paths of depth 3: 6 paths
	Paths of depth 4: 54 paths
	Paths of depth 5: 63 paths
	Paths of depth 6: 89 paths
	Paths of depth 7: 137 paths
	Paths of depth 8: 158 paths
	Paths of depth 9: 191 paths
	Paths of depth 10: 204 paths
	Paths of depth 11: 248 paths
	Paths of depth 12: 552 paths



Trained on depth 15
	Paths of depth 2: 1 paths
	Paths of depth 3: 5 paths
	Paths of depth 4: 52 paths
	Paths of depth 5: 61 paths
	Paths of depth 6: 85 paths
	Paths of depth 7: 158 paths
	Paths of depth 8: 141 paths
	Paths of depth 9: 192 paths
	Paths of depth 10: 215 paths
	Paths of depth 11: 269 paths
	Paths of depth 12: 265 paths
	Paths of depth 13: 290 paths
	Paths of depth 14: 319 paths
	Paths of depth 15: 642 paths



Trained on depth 18
	Paths of depth 2: 1 paths
	Paths of depth 3: 7 paths
	Paths of depth 4: 50 paths
	Paths of depth 5: 63 paths
	Paths of depth 6: 74 paths
	Paths of depth 7: 151 paths
	Paths of depth 8: 161 paths
	Paths of depth 9: 176 paths
	Paths of dep

In [ ]:
gbm_1trees = different_depth_lightgbm_models(transactions_train[train_features], transactions_train['isFraud'], LIGHTGBM_PARAMS, num_rounds=1, depths=[12,15,18, 21])




Trained on depth 12
	Paths of depth 3: 3 paths
	Paths of depth 4: 8 paths
	Paths of depth 6: 2 paths
	Paths of depth 7: 3 paths
	Paths of depth 8: 6 paths
	Paths of depth 9: 10 paths
	Paths of depth 10: 11 paths
	Paths of depth 11: 19 paths
	Paths of depth 12: 30 paths



Trained on depth 15
	Paths of depth 3: 3 paths
	Paths of depth 4: 8 paths
	Paths of depth 6: 2 paths
	Paths of depth 7: 3 paths
	Paths of depth 8: 6 paths
	Paths of depth 9: 10 paths
	Paths of depth 10: 11 paths
	Paths of depth 11: 19 paths
	Paths of depth 12: 13 paths
	Paths of depth 13: 17 paths
	Paths of depth 14: 19 paths
	Paths of depth 15: 30 paths



Trained on depth 18
	Paths of depth 3: 3 paths
	Paths of depth 4: 8 paths
	Paths of depth 6: 2 paths
	Paths of depth 7: 3 paths
	Paths of depth 8: 6 paths
	Paths of depth 9: 10 paths
	Paths of depth 10: 11 paths
	Paths of depth 11: 19 paths
	Paths of depth 12: 13 paths
	Paths of depth 13: 17 paths
	Paths of depth 14: 19 paths
	Paths of depth 15: 11 paths
	Paths 

In [ ]:
def high_depth_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, background_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf.high_depth_woodelf.woodelf_for_high_depth(model, consumer_data, background_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [ ]:
def simple_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, background_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False,
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf.simple_woodelf.calculate_background_metric(model, consumer_data, background_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [ ]:
def shap_on_models_dict(models, consumer_data: pd.DataFrame, background_data: pd.DataFrame):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        explainer = shap.TreeExplainer(model, background_data, feature_perturbation='interventional')
        simple_shap_values = explainer.shap_values(consumer_data)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

# Background SHAP

In [ ]:
woodelf_running_times = high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_100trees[12], 15: gbm_100trees[15], 18: gbm_100trees[18], 21: gbm_10trees[21]},
    consumer_data=transactions_test[train_features], background_data=transactions_train[train_features], metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:14<00:00,  6.98it/s]


M time: 0.0 sec, s time: 0.46 sec (f prepare time: 0.15880489349365234)
On Depth 6 Took: 14.437673807144165


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:46<00:00,  2.16it/s]


M time: 0.01 sec, s time: 1.99 sec (f prepare time: 0.576153039932251)
On Depth 9 Took: 46.527263164520264


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:44<00:00,  1.05s/it]


...

M time: 1.21 sec, s time: 88.52 sec (f prepare time: 6.905958890914917)
On Depth 15 Took: 285.4658627510071


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [23:34<00:00, 14.14s/it]


M time: 11.73 sec, s time: 1052.39 sec (f prepare time: 44.19594621658325)
On Depth 18 Took: 1427.3464524745941


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [29:04<00:00, 174.49s/it]

M time: 103.15 sec, s time: 1590.61 sec (f prepare time: 66.03927063941956)
On Depth 21 Took: 1848.3017585277557
Background SHAP: 14.437673807144165 & 46.527263164520264 & 105.28893494606018 & 285.4658627510071 & 1427.3464524745941 & 1848.3017585277557


In [ ]:
woodelf_running_times = simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_10trees[12], 15: gbm_1trees[15]}, # Depth 18 crashes due to RAM
    consumer_data=transactions_test[train_features], background_data=transactions_train[train_features], metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:05<00:00, 17.04it/s]


cache misses: 71, cache used: 2973, M computation time: 0.17 sec, s computation time: 0.15 sec


Computing the values: 100%|██████████| 100/100 [00:06<00:00, 16.11it/s]


On Depth 6 Took: 12.200419187545776


Preprocessing the trees: 100%|██████████| 100/100 [00:58<00:00,  1.71it/s]


...

On Depth 12 Took: 350.22402787208557


Preprocessing the trees: 100%|██████████| 1/1 [21:06<00:00, 1266.25s/it]


cache misses: 29, cache used: 112, M computation time: 1236.2 sec, s computation time: 10.07 sec


Computing the values: 100%|██████████| 1/1 [00:03<00:00,  3.51s/it]

On Depth 15 Took: 1269.883497953415
Background SHAP: 12.200419187545776 & 82.74372887611389 & 350.22402787208557 & 1269.883497953415


In [ ]:
train_sample = fraud_train.sample(10, random_state=42)

shap_running_times = shap_on_models_dict(
    gbm_100trees, consumer_data=fraud_test, background_data=train_sample,
)
print("Background SHAP: " + " & ".join([str(shap_running_times[d]) for d in sorted(list(shap_running_times.keys()))]))

100%|===================| 117981/118108 [00:58<00:00]       

On Depth 6 Took: 61.20456600189209


 99%|===================| 117232/118108 [01:49<00:00]       

On Depth 9 Took: 111.20987391471863


100%|===================| 118041/118108 [02:52<00:00]       

On Depth 12 Took: 174.71757745742798


100%|===================| 117795/118108 [03:59<00:00]       

On Depth 15 Took: 242.83412337303162


100%|===================| 117984/118108 [05:10<00:00]       

On Depth 18 Took: 314.1569471359253
Background SHAP: 61.20456600189209 & 111.20987391471863 & 174.71757745742798 & 242.83412337303162 & 314.1569471359253


# Background SHAP IV

In [ ]:
woodelf_running_times = high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_100trees[12], 15: gbm_100trees[15], 18: gbm_10trees[18]}, # depth 21 crashes due to RAM
    consumer_data=transactions_test[train_features].sample(10_000, random_state=42), background_data=transactions_train[train_features], metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:08<00:00, 11.82it/s]


M time: 0.0 sec, s time: 0.54 sec (f prepare time: 0.14290881156921387)
On Depth 6 Took: 8.585093021392822


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:31<00:00,  3.22it/s]


M time: 0.02 sec, s time: 3.08 sec (f prepare time: 0.5433657169342041)
On Depth 9 Took: 31.38403081893921


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:50<00:00,  1.10s/it]


M time: 0.32 sec, s time: 42.31 sec (f prepare time: 1.7553761005401611)
On Depth 12 Took: 110.88345837593079


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [15:11<00:00,  9.11s/it]


M time: 4.58 sec, s time: 742.18 sec (f prepare time: 6.955220699310303)
On Depth 15 Took: 916.4175143241882


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [25:57<00:00, 155.74s/it]

M time: 55.58 sec, s time: 1396.2 sec (f prepare time: 8.042815685272217)
On Depth 18 Took: 1613.155867099762
Background SHAP: 8.585093021392822 & 31.38403081893921 & 110.88345837593079 & 916.4175143241882 & 1613.155867099762


In [ ]:
woodelf_running_times = simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_10trees[12]}, # depth 15 crashes due to RAM
    consumer_data=transactions_test[train_features].sample(10_000, random_state=42), background_data=transactions_train[train_features], metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:06<00:00, 15.59it/s]


cache misses: 71, cache used: 2973, M computation time: 0.27 sec, s computation time: 0.34 sec


Computing the values: 100%|██████████| 100/100 [00:01<00:00, 54.14it/s]


On Depth 6 Took: 8.417188882827759


Preprocessing the trees: 100%|██████████| 100/100 [01:26<00:00,  1.15it/s]


...

On Depth 9 Took: 102.97197151184082


Preprocessing the trees: 100%|██████████| 10/10 [14:41<00:00, 88.11s/it]


cache misses: 179, cache used: 1523, M computation time: 854.94 sec, s computation time: 16.37 sec


Computing the values: 100%|██████████| 10/10 [00:24<00:00,  2.47s/it]

On Depth 12 Took: 905.8875668048859
Background SHAP: 8.417188882827759 & 102.97197151184082 & 905.8875668048859


In [ ]:
# shap Package does not support Background SHAP IV

# RAM Crashes

In [ ]:
# Crashed due to RAM

woodelf_running_times = simple_woodelf_on_models_dict(
    {18: gbm_1trees[18]},
    consumer_data=transactions_test[train_features], background_data=transactions_train[train_features], metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Crashed due to RAM

woodelf_running_times = simple_woodelf_on_models_dict(
    {15: gbm_1trees[15]},
    consumer_data=transactions_test[train_features], background_data=transactions_train[train_features], metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Crashed due to RAM (in the HighDepthPathToMatrices.build_matrices phase)

woodelf_running_times = high_depth_woodelf_on_models_dict(
    {21: gbm_1trees[21]},
    consumer_data=transactions_test[train_features].sample(10_000, random_state=42), background_data=transactions_train[train_features], metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))